# Spark Assignment-6(part-1)

## Objective

The objective of this assignment is to understand Spark architecture and perform data processing using DataFrames. The assignment covers reading data from different file formats, applying transformations, filtering records, handling schema and null values, understanding lazy evaluation, and writing processed data back into CSV and Parquet formats.


## Understanding Spark Architecture

Apache Spark follows a distributed architecture where the Driver Program creates and manages Spark applications. The Driver communicates with the Cluster Manager to request resources.

The Cluster Manager allocates resources and launches Executors on different worker nodes. Executors perform the actual computations and return the results to the Driver.

Main Components:

- Driver - Creates SparkSession and coordinates execution.
- Cluster Manager – Allocates resources.
- Worker Nodes – Machines that execute tasks.
- Executors – Run tasks and store intermediate data.

This architecture allows Spark to process large datasets efficiently in parallel.

## Spark Execution Modes

Spark applications can run in different execution modes depending on the environment.

- Local Mode – Runs on a single machine and is mainly used for learning and testing.
- Standalone Cluster – Spark manages its own cluster.
- Hadoop YARN – Uses Hadoop for resource management.
- Kubernetes – Runs Spark applications inside Kubernetes clusters.

Hera Databricks manages the Spark cluster automatically.

## Reading the CSV File

The first step is to load the student dataset into a Spark DataFrame. Here I am using the `inferSchema` option so Spark automatically detects the data type of each column. The `header=True` option tells Spark to use the first row as column names.

In [0]:
# No need for session data briscks automatically creates it for us
df = (spark.read.option("header", True).option("inferSchema", True).csv("/Volumes/workspace/default/my_files/Student.csv"))
df.show(10)

+---------+-------------+----------+----+------+---+----------+-----+----+---------+-----------+--------------------+
|StudentID|         Name|Department|Year|Gender|Age|Attendance|Marks|CGPA|     City|Scholarship|               Email|
+---------+-------------+----------+----+------+---+----------+-----+----+---------+-----------+--------------------+
|     1001| Aarav Sharma|        CS|   1|     M| 18|        92|   85| 8.8|    Delhi|        Yes|aarav.sharma@univ...|
|     1002|    Aditi Rao|       ECE|   2|     F| 19|        88|   78| 8.2|   Mumbai|         No|  aditi.rao@univ.edu|
|     1003|  Arjun Verma|        ME|   3|     M| 21|        74|   65| 7.1|Bangalore|         No|arjun.verma@univ.edu|
|     1004|  Ananya Iyer|        CS|   1|     F| 18|        95|   92| 9.4|  Chennai|        Yes|ananya.iyer@univ.edu|
|     1005| Aditya Patel|        IT|   4|     M| 22|        68|   58| 6.5|Ahmedabad|         No|aditya.patel@univ...|
|     1006|  Avani Singh|       ECE|   2|     F| 20|    

## Checking the Schema

After loading the data, I verified the schema to ensure that Spark identified the correct data types for each column..

In [0]:
df.printSchema()

root
 |-- StudentID: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Attendance: string (nullable = true)
 |-- Marks: string (nullable = true)
 |-- CGPA: double (nullable = true)
 |-- City: string (nullable = true)
 |-- Scholarship: string (nullable = true)
 |-- Email: string (nullable = true)



## Reading Data with an Explicit Schema

Instead of  automatic schema detection, Spark also allows us to define the schema manually. This improves performance because Spark does not need to scan the data to determine the data types.

In [0]:
# Column Name || Type of the col data || isnullable 
from pyspark.sql.types import *


student_schema = StructType([StructField("StudentID", IntegerType(), True),
StructField("Name", StringType(), True),StructField("Department", StringType(), True),StructField("Year", IntegerType(), True),StructField("Gender", StringType(), True),StructField("Age", IntegerType(), True),StructField("Attendance", IntegerType(), True),StructField("Marks", IntegerType(), True),StructField("CGPA", DoubleType(), True),StructField("City", StringType(), True),StructField("Scholarship", StringType(), True),StructField("Email", StringType(), True)])
student_df = (spark.read.option("header", True).schema(student_schema).csv("/Volumes/workspace/default/my_files/Student.csv")
)

student_df.printSchema()

root
 |-- StudentID: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Attendance: integer (nullable = true)
 |-- Marks: integer (nullable = true)
 |-- CGPA: double (nullable = true)
 |-- City: string (nullable = true)
 |-- Scholarship: string (nullable = true)
 |-- Email: string (nullable = true)



## Saving Data as a Parquet File

After reading the CSV file, I saved the DataFrame in Parquet format.

Parquet is a columnar storage format that stores data efficiently. It occupies less storage space and provides faster query performance compared to CSV because only the required columns are read during execution.

In [0]:
student_df.write.mode("overwrite").parquet("/Volumes/workspace/default/my_files/student_parquet")

## Reading the Parquet File

After creating the Parquet file, I loaded it back into a DataFrame. Spark automatically reads the schema stored inside the Parquet file, so there is no need to use inferSchema or manually define the schema.

In [0]:
parquet_df = spark.read.parquet("/Volumes/workspace/default/my_files/student_parquet")

parquet_df.show(10)

+---------+-------------+----------+----+------+---+----------+-----+----+---------+-----------+--------------------+
|StudentID|         Name|Department|Year|Gender|Age|Attendance|Marks|CGPA|     City|Scholarship|               Email|
+---------+-------------+----------+----+------+---+----------+-----+----+---------+-----------+--------------------+
|     1001| Aarav Sharma|        CS|   1|     M| 18|        92|   85| 8.8|    Delhi|        Yes|aarav.sharma@univ...|
|     1002|    Aditi Rao|       ECE|   2|     F| 19|        88|   78| 8.2|   Mumbai|         No|  aditi.rao@univ.edu|
|     1003|  Arjun Verma|        ME|   3|     M| 21|        74|   65| 7.1|Bangalore|         No|arjun.verma@univ.edu|
|     1004|  Ananya Iyer|        CS|   1|     F| 18|        95|   92| 9.4|  Chennai|        Yes|ananya.iyer@univ.edu|
|     1005| Aditya Patel|        IT|   4|     M| 22|        68|   58| 6.5|Ahmedabad|         No|aditya.patel@univ...|
|     1006|  Avani Singh|       ECE|   2|     F| 20|    

In [0]:
parquet_df.printSchema()

root
 |-- StudentID: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Attendance: integer (nullable = true)
 |-- Marks: integer (nullable = true)
 |-- CGPA: double (nullable = true)
 |-- City: string (nullable = true)
 |-- Scholarship: string (nullable = true)
 |-- Email: string (nullable = true)



## CSV vs Parquet

Although both formats store the same data, they are designed differently.

CSV stores data row by row and does not keep schema information.

Parquet stores data column by column along with metadata. Because of this, Spark can read only the required columns instead of scanning the entire file.

This makes Parquet much faster and more suitable for big data applications.

In [0]:
student_df.select("Name", "Marks").show(10)

+-------------+-----+
|         Name|Marks|
+-------------+-----+
| Aarav Sharma|   85|
|    Aditi Rao|   78|
|  Arjun Verma|   65|
|  Ananya Iyer|   92|
| Aditya Patel|   58|
|  Avani Singh|   74|
|  Ayush Gupta|   88|
|  Amrita Nair|   81|
|    Akash Das|   62|
|Anjali Mishra| NULL|
+-------------+-----+
only showing top 10 rows


In [0]:
parquet_df.select("Name", "Marks").show(10)

+-------------+-----+
|         Name|Marks|
+-------------+-----+
| Aarav Sharma|   85|
|    Aditi Rao|   78|
|  Arjun Verma|   65|
|  Ananya Iyer|   92|
| Aditya Patel|   58|
|  Avani Singh|   74|
|  Ayush Gupta|   88|
|  Amrita Nair|   81|
|    Akash Das|   62|
|Anjali Mishra| NULL|
+-------------+-----+
only showing top 10 rows


### Observation

The output is the same for both DataFrames. However, Parquet performs better internally because Spark reads only the selected columns instead of scanning the complete dataset.

## Predicate Pushdown

Predicate Pushdown is an optimization used by Spark when reading formats like Parquet.

If a filter condition is applied while reading data, Spark pushes that condition closer to the storage layer. As a result, only the required records are read instead of loading the complete dataset.

This reduces disk I/O and improves query performance.

CSV files do not support Predicate Pushdown effectively because Spark has to scan the file line by line.

In [0]:
parquet_df.filter(parquet_df.Marks > 80).show(10)

+---------+-------------+----------+----+------+---+----------+-----+----+---------+-----------+--------------------+
|StudentID|         Name|Department|Year|Gender|Age|Attendance|Marks|CGPA|     City|Scholarship|               Email|
+---------+-------------+----------+----+------+---+----------+-----+----+---------+-----------+--------------------+
|     1001| Aarav Sharma|        CS|   1|     M| 18|        92|   85| 8.8|    Delhi|        Yes|aarav.sharma@univ...|
|     1004|  Ananya Iyer|        CS|   1|     F| 18|        95|   92| 9.4|  Chennai|        Yes|ananya.iyer@univ.edu|
|     1007|  Ayush Gupta|        CS|   3|     M| 21|        89|   88| 8.9|    Delhi|        Yes|ayush.gupta@univ.edu|
|     1008|  Amrita Nair|        EE|   1|     F| 18|        90|   81| 8.3|    Kochi|         No|amrita.nair@univ.edu|
|     1011| Bhavya Joshi|        CS|   4|     F| 22|        94|   95| 9.6|     Pune|        Yes|bhavya.joshi@univ...|
|     1015|   Devika Sen|        CS|   2|     F| 19|    

## Selecting Required Columns

In many real-world projects, we don't need every column from a dataset. Selecting only the required columns reduces memory usage and improves readability of the DataFrame.

In [0]:
student_df.select("StudentID","Name","Department","Marks","CGPA").show(10)

+---------+-------------+----------+-----+----+
|StudentID|         Name|Department|Marks|CGPA|
+---------+-------------+----------+-----+----+
|     1001| Aarav Sharma|        CS|   85| 8.8|
|     1002|    Aditi Rao|       ECE|   78| 8.2|
|     1003|  Arjun Verma|        ME|   65| 7.1|
|     1004|  Ananya Iyer|        CS|   92| 9.4|
|     1005| Aditya Patel|        IT|   58| 6.5|
|     1006|  Avani Singh|       ECE|   74| 7.8|
|     1007|  Ayush Gupta|        CS|   88| 8.9|
|     1008|  Amrita Nair|        EE|   81| 8.3|
|     1009|    Akash Das|        ME|   62| 6.9|
|     1010|Anjali Mishra|        IT| NULL| 7.6|
+---------+-------------+----------+-----+----+
only showing top 10 rows


## Filtering Data

Filtering is used to retrieve only those rows that satisfy a particular condition.

In [0]:
student_df.filter(student_df.Marks > 80).show(10)

+---------+-------------+----------+----+------+---+----------+-----+----+---------+-----------+--------------------+
|StudentID|         Name|Department|Year|Gender|Age|Attendance|Marks|CGPA|     City|Scholarship|               Email|
+---------+-------------+----------+----+------+---+----------+-----+----+---------+-----------+--------------------+
|     1001| Aarav Sharma|        CS|   1|     M| 18|        92|   85| 8.8|    Delhi|        Yes|aarav.sharma@univ...|
|     1004|  Ananya Iyer|        CS|   1|     F| 18|        95|   92| 9.4|  Chennai|        Yes|ananya.iyer@univ.edu|
|     1007|  Ayush Gupta|        CS|   3|     M| 21|        89|   88| 8.9|    Delhi|        Yes|ayush.gupta@univ.edu|
|     1008|  Amrita Nair|        EE|   1|     F| 18|        90|   81| 8.3|    Kochi|         No|amrita.nair@univ.edu|
|     1011| Bhavya Joshi|        CS|   4|     F| 22|        94|   95| 9.6|     Pune|        Yes|bhavya.joshi@univ...|
|     1015|   Devika Sen|        CS|   2|     F| 19|    

In [0]:
student_df.filter(student_df.Department == "CS").show(10)

+---------+-------------+----------+----+------+---+----------+-----+----+---------+-----------+--------------------+
|StudentID|         Name|Department|Year|Gender|Age|Attendance|Marks|CGPA|     City|Scholarship|               Email|
+---------+-------------+----------+----+------+---+----------+-----+----+---------+-----------+--------------------+
|     1001| Aarav Sharma|        CS|   1|     M| 18|        92|   85| 8.8|    Delhi|        Yes|aarav.sharma@univ...|
|     1004|  Ananya Iyer|        CS|   1|     F| 18|        95|   92| 9.4|  Chennai|        Yes|ananya.iyer@univ.edu|
|     1007|  Ayush Gupta|        CS|   3|     M| 21|        89|   88| 8.9|    Delhi|        Yes|ayush.gupta@univ.edu|
|     1011| Bhavya Joshi|        CS|   4|     F| 22|        94|   95| 9.6|     Pune|        Yes|bhavya.joshi@univ...|
|     1015|   Devika Sen|        CS|   2|     F| 19|        91|   87| 8.7|  Kolkata|        Yes| devika.sen@univ.edu|
|     1018|   Divya Teja|        CS|   3|     F| 20|    

In [0]:
student_df.filter(student_df.Attendance >= 90).show(10)

+---------+-------------+----------+----+------+---+----------+-----+----+---------+-----------+--------------------+
|StudentID|         Name|Department|Year|Gender|Age|Attendance|Marks|CGPA|     City|Scholarship|               Email|
+---------+-------------+----------+----+------+---+----------+-----+----+---------+-----------+--------------------+
|     1001| Aarav Sharma|        CS|   1|     M| 18|        92|   85| 8.8|    Delhi|        Yes|aarav.sharma@univ...|
|     1004|  Ananya Iyer|        CS|   1|     F| 18|        95|   92| 9.4|  Chennai|        Yes|ananya.iyer@univ.edu|
|     1008|  Amrita Nair|        EE|   1|     F| 18|        90|   81| 8.3|    Kochi|         No|amrita.nair@univ.edu|
|     1011| Bhavya Joshi|        CS|   4|     F| 22|        94|   95| 9.6|     Pune|        Yes|bhavya.joshi@univ...|
|     1015|   Devika Sen|        CS|   2|     F| 19|        91|   87| 8.7|  Kolkata|        Yes| devika.sen@univ.edu|
|     1018|   Divya Teja|        CS|   3|     F| 20|    

## Combining Selection and Filtering

Spark allows multiple DataFrame operations to be chained together. This makes the code more readable and avoids creating unnecessary intermediate DataFrames.

In [0]:
student_df.filter(student_df.Marks >= 75).select("Name","Department","Marks").show(10)

+--------------+----------+-----+
|          Name|Department|Marks|
+--------------+----------+-----+
|  Aarav Sharma|        CS|   85|
|     Aditi Rao|       ECE|   78|
|   Ananya Iyer|        CS|   92|
|   Ayush Gupta|        CS|   88|
|   Amrita Nair|        EE|   81|
|  Bhavya Joshi|        CS|   95|
|    Devika Sen|        CS|   87|
|  Dhruv Saxena|       ECE|   80|
|    Divya Teja|        CS|   91|
|Esha Choudhury|        IT|   76|
+--------------+----------+-----+
only showing top 10 rows


## Renaming Columns
 Spark provides the `withColumnRenamed()` function to rename columns.

In [0]:
student_df.withColumnRenamed("CGPA","Current_CGPA").show(10)

+---------+-------------+----------+----+------+---+----------+-----+------------+---------+-----------+--------------------+
|StudentID|         Name|Department|Year|Gender|Age|Attendance|Marks|Current_CGPA|     City|Scholarship|               Email|
+---------+-------------+----------+----+------+---+----------+-----+------------+---------+-----------+--------------------+
|     1001| Aarav Sharma|        CS|   1|     M| 18|        92|   85|         8.8|    Delhi|        Yes|aarav.sharma@univ...|
|     1002|    Aditi Rao|       ECE|   2|     F| 19|        88|   78|         8.2|   Mumbai|         No|  aditi.rao@univ.edu|
|     1003|  Arjun Verma|        ME|   3|     M| 21|        74|   65|         7.1|Bangalore|         No|arjun.verma@univ.edu|
|     1004|  Ananya Iyer|        CS|   1|     F| 18|        95|   92|         9.4|  Chennai|        Yes|ananya.iyer@univ.edu|
|     1005| Aditya Patel|        IT|   4|     M| 22|        68|   58|         6.5|Ahmedabad|         No|aditya.patel@u

## Casting Data Types

Sometimes the data type of a column needs to be changed before performing calculations or analysis. Spark provides the `cast()` function for this purpose.

In [0]:
# Using col for selscting columns is good paractice
from pyspark.sql.functions import col
cast_df = student_df.withColumn("Age",col("Age").cast("double"))

cast_df.printSchema()

root
 |-- StudentID: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- Attendance: integer (nullable = true)
 |-- Marks: integer (nullable = true)
 |-- CGPA: double (nullable = true)
 |-- City: string (nullable = true)
 |-- Scholarship: string (nullable = true)
 |-- Email: string (nullable = true)



## Creating a New Column

Spark allows new columns to be created based on existing column values. This is useful for deriving additional information from the dataset.

In [0]:
# when is like using if else conditions 
from pyspark.sql.functions import when
student_df.withColumn("Result",when(col("Marks") >= 40, "Pass").otherwise("Fail")).show(10)


+---------+-------------+----------+----+------+---+----------+-----+----+---------+-----------+--------------------+------+
|StudentID|         Name|Department|Year|Gender|Age|Attendance|Marks|CGPA|     City|Scholarship|               Email|Result|
+---------+-------------+----------+----+------+---+----------+-----+----+---------+-----------+--------------------+------+
|     1001| Aarav Sharma|        CS|   1|     M| 18|        92|   85| 8.8|    Delhi|        Yes|aarav.sharma@univ...|  Pass|
|     1002|    Aditi Rao|       ECE|   2|     F| 19|        88|   78| 8.2|   Mumbai|         No|  aditi.rao@univ.edu|  Pass|
|     1003|  Arjun Verma|        ME|   3|     M| 21|        74|   65| 7.1|Bangalore|         No|arjun.verma@univ.edu|  Pass|
|     1004|  Ananya Iyer|        CS|   1|     F| 18|        95|   92| 9.4|  Chennai|        Yes|ananya.iyer@univ.edu|  Pass|
|     1005| Aditya Patel|        IT|   4|     M| 22|        68|   58| 6.5|Ahmedabad|         No|aditya.patel@univ...|  Pass|


## Transformations and Actions

Spark operations are divided into two categories: Transformations and Actions.

A transformation creates a new DataFrame from an existing one, but Spark does not execute it immediately. Instead, Spark remembers the sequence of transformations.

An action triggers the execution of all pending transformations and returns the result.

In [0]:
# Nothing wil happed yet
#no output will be produced 
filtered_students = (student_df.filter(col("Marks") > 70).select("Name", "Department", "Marks"))

In [0]:
# O/p got coz it action is performed
filtered_students.show(10)

+------------+----------+-----+
|        Name|Department|Marks|
+------------+----------+-----+
|Aarav Sharma|        CS|   85|
|   Aditi Rao|       ECE|   78|
| Ananya Iyer|        CS|   92|
| Avani Singh|       ECE|   74|
| Ayush Gupta|        CS|   88|
| Amrita Nair|        EE|   81|
|Bhavya Joshi|        CS|   95|
|Chetan Kumar|        IT|   72|
|  Devika Sen|        CS|   87|
|Dhruv Saxena|       ECE|   80|
+------------+----------+-----+
only showing top 10 rows


## Examples of Transformations

Some commonly used DataFrame transformations are:

- select()
- filter()
- withColumn()
- withColumnRenamed()
- drop()
- groupBy()
- orderBy()

These transformations are lazy and return a new DataFrame.

## Examples of Actions

Actions trigger the execution of transformations and produce results.

In [0]:
student_df.show(2)

+---------+------------+----------+----+------+---+----------+-----+----+------+-----------+--------------------+
|StudentID|        Name|Department|Year|Gender|Age|Attendance|Marks|CGPA|  City|Scholarship|               Email|
+---------+------------+----------+----+------+---+----------+-----+----+------+-----------+--------------------+
|     1001|Aarav Sharma|        CS|   1|     M| 18|        92|   85| 8.8| Delhi|        Yes|aarav.sharma@univ...|
|     1002|   Aditi Rao|       ECE|   2|     F| 19|        88|   78| 8.2|Mumbai|         No|  aditi.rao@univ.edu|
+---------+------------+----------+----+------+---+----------+-----+----+------+-----------+--------------------+
only showing top 2 rows


In [0]:
student_df.count()

100

In [0]:
student_df.first()

Row(StudentID=1001, Name='Aarav Sharma', Department='CS', Year=1, Gender='M', Age=18, Attendance=92, Marks=85, CGPA=8.8, City='Delhi', Scholarship='Yes', Email='aarav.sharma@univ.edu')

### dangerous command not preferred to use with huge datasets 
- student_df.collect()

### Observation

Unlike transformations, actions execute the complete logical plan and return the output.

## Lazy Evaluation

Spark follows the concept of Lazy Evaluation.

Instead of executing every transformation immediately, Spark stores all transformations and waits until an action is called.

This allows Spark to optimize the execution plan and reduce unnecessary computations.

### Observation

The transformations are executed only after calling `show()`. Spark combines all transformations into an optimized execution plan.

## Directed Acyclic Graph (DAG)

Spark represents every transformation as a Directed Acyclic Graph (DAG).

Each transformation becomes a node in the graph, and Spark analyzes the complete graph before execution.

This helps Spark optimize the execution plan by combining operations and removing unnecessary work.

## Narrow and Wide Transformations

Transformations in Spark are classified as Narrow or Wide based on how data is processed.

A Narrow Transformation processes data within the same partition.

A Wide Transformation requires data movement between partitions, which causes a shuffle.

In [0]:
student_df.filter(col("Marks") > 70).show(10)

+---------+------------+----------+----+------+---+----------+-----+----+---------+-----------+--------------------+
|StudentID|        Name|Department|Year|Gender|Age|Attendance|Marks|CGPA|     City|Scholarship|               Email|
+---------+------------+----------+----+------+---+----------+-----+----+---------+-----------+--------------------+
|     1001|Aarav Sharma|        CS|   1|     M| 18|        92|   85| 8.8|    Delhi|        Yes|aarav.sharma@univ...|
|     1002|   Aditi Rao|       ECE|   2|     F| 19|        88|   78| 8.2|   Mumbai|         No|  aditi.rao@univ.edu|
|     1004| Ananya Iyer|        CS|   1|     F| 18|        95|   92| 9.4|  Chennai|        Yes|ananya.iyer@univ.edu|
|     1006| Avani Singh|       ECE|   2|     F| 20|        81|   74| 7.8|  Lucknow|         No|avani.singh@univ.edu|
|     1007| Ayush Gupta|        CS|   3|     M| 21|        89|   88| 8.9|    Delhi|        Yes|ayush.gupta@univ.edu|
|     1008| Amrita Nair|        EE|   1|     F| 18|        90|  

In [0]:
student_df.groupBy("Department").count().show()

+----------+-----+
|Department|count|
+----------+-----+
|        EE|   17|
|       ECE|   20|
|        IT|   20|
|        CS|   24|
|        ME|   19|
+----------+-----+



### Observation

Filtering and selecting columns are narrow transformations because they do not move data across partitions.

Operations like groupBy() require data from multiple partitions to be combined, resulting in a shuffle.

## Shuffle

Shuffle is the process of redistributing data between partitions.

Operations such as groupBy(), join(), distinct(), and orderBy() usually trigger a shuffle.

Since data is exchanged across the cluster, shuffle is one of the most expensive operations in Spark.

## Best Practices

While working with large datasets, it is important to follow some best practices.

- Select only the required columns.
- Apply filters as early as possible.
- Use Parquet instead of CSV whenever possible.
- Avoid unnecessary shuffle operations.
- Avoid using collect() on large datasets.
- Use show() for inspecting data.
- Define the schema explicitly instead of using inferSchema in production.

## Building a Data Processing Pipeline

A data pipeline is a sequence of operations performed on a dataset.

In this example, the data is read from a CSV file, transformed by creating a new column, filtered based on marks, and finally written to storage.

In [0]:
from pyspark.sql.functions import when, col

df = spark.read.option("header", True).schema(student_schema).csv("/Volumes/workspace/default/my_files/Student.csv");pipeline_df=df.withColumn("Result", when(col("Marks") >= 40, "Pass").otherwise("Fail")).filter(col("Marks") >= 60).select("StudentID", "Name", "Department", "Marks", "Result")

pipeline_df.show(10)

+---------+------------+----------+-----+------+
|StudentID|        Name|Department|Marks|Result|
+---------+------------+----------+-----+------+
|     1001|Aarav Sharma|        CS|   85|  Pass|
|     1002|   Aditi Rao|       ECE|   78|  Pass|
|     1003| Arjun Verma|        ME|   65|  Pass|
|     1004| Ananya Iyer|        CS|   92|  Pass|
|     1006| Avani Singh|       ECE|   74|  Pass|
|     1007| Ayush Gupta|        CS|   88|  Pass|
|     1008| Amrita Nair|        EE|   81|  Pass|
|     1009|   Akash Das|        ME|   62|  Pass|
|     1011|Bhavya Joshi|        CS|   95|  Pass|
|     1012|Bharat Reddy|       ECE|   69|  Pass|
+---------+------------+----------+-----+------+
only showing top 10 rows


CSV File->Read Data->Transform Data->Filter Records->Select Required Columns->Write Output

### Saving the final dataframe into csv format

In [0]:
pipeline_df.write.mode("overwrite").option("header", True).csv("/Volumes/workspace/default/my_files/output_csv")